# Drop stale Bronze tables

One-off maintenance notebook. It is **not** part of `tourism_pipeline` and should only be run by hand.

Drops 8 stale tables from `tuklas_dev.02_bronze` to free Unity Catalog metastore quota (the metastore was at its 500-table limit on 2026-10-20).

**Stale tables:** early PSGC Bronze tables written with a `psa_psgc_` prefix, before PSGC ingestion moved to `02_ingest_psgc` and the `psgc_` prefix. No notebook on `main` writes or reads them, and lineage showed no downstream consumers on 2026-10-20.

| # | Stale table | Replaced by |
| --- | --- | --- |
| 1 | `psa_psgc_report_text` | `psgc_report_text` |
| 2 | `psa_psgc_source_files` | `psgc_source_files` |
| 3 | `psa_psgc_ds_33_provincial_summary` | `psgc_ds_33_provincial_summary` |
| 4 | `psa_psgc_ds_34_metadata` | `psgc_ds_34_metadata` |
| 5 | `psa_psgc_ds_34_psgc` | `psgc_ds_34_psgc` |
| 6 | `psa_psgc_ds_34_notes` | `psgc_ds_34_notes` |
| 7 | `psa_psgc_ds_34_national_summary` | `psgc_ds_34_national_summary` |
| 8 | `psa_psgc_ds_34_prov_sum` | `psgc_ds_34_prov_sum` |

**Not dropped: `psa_ds_18_pdf`.** It looks like a duplicate of `psa_ds_18_pdf_rows`, but it isn't. `01_ingest_psa` writes it on every pipeline run (the DS_18 PDF stored as binary, with its SHA-256), while `psa_ds_18_pdf_rows` holds the parsed rows from `04_ingest_psa_ds18_pdf`. Dropping it would only free quota until the next run recreates it.

**How to run**

1. **Pre-flight:** all 8 stale tables and all 8 replacements must be listed (16 rows).
2. **Safety check:** must return **zero rows**: no downstream consumers.
3. Set `DROP_CONFIRMED = True` in the drop cell and run it. It stops with an error if any drop fails.
4. **Post-check:** `02_bronze` should go from 58 to 50 tables.

Dropped managed tables can be restored with `UNDROP TABLE` for a limited time (7 days by default), but don't rely on it: check steps 1 and 2 first.


In [ ]:
%sql
-- Pre-flight: the 8 stale tables still exist AND their replacements are in place.
-- Expected: 16 rows, 8 STALE and 8 REPLACEMENT. If any row is missing, stop.
SELECT
  table_name,
  created,
  CASE WHEN table_name LIKE 'psa_psgc_%' THEN 'STALE' ELSE 'REPLACEMENT' END AS role
FROM system.information_schema.tables
WHERE table_catalog = 'tuklas_dev'
  AND table_schema = '02_bronze'
  AND table_name IN (
    -- stale
    'psa_psgc_report_text',
    'psa_psgc_source_files',
    'psa_psgc_ds_33_provincial_summary',
    'psa_psgc_ds_34_metadata',
    'psa_psgc_ds_34_psgc',
    'psa_psgc_ds_34_notes',
    'psa_psgc_ds_34_national_summary',
    'psa_psgc_ds_34_prov_sum',
    -- replacements
    'psgc_report_text',
    'psgc_source_files',
    'psgc_ds_33_provincial_summary',
    'psgc_ds_34_metadata',
    'psgc_ds_34_psgc',
    'psgc_ds_34_notes',
    'psgc_ds_34_national_summary',
    'psgc_ds_34_prov_sum'
  )
ORDER BY role, table_name

In [ ]:
%sql
-- Safety check: any downstream consumers of the 8 stale tables?
-- Expected: ZERO rows. If any rows appear, something still reads a stale table: fix that first.
SELECT
  source_table_name,
  target_table_catalog || '.' || target_table_schema || '.' || target_table_name AS downstream_target,
  entity_type,
  event_time
FROM system.access.table_lineage
WHERE source_table_catalog = 'tuklas_dev'
  AND source_table_schema = '02_bronze'
  AND source_table_name IN (
    'psa_psgc_report_text',
    'psa_psgc_source_files',
    'psa_psgc_ds_33_provincial_summary',
    'psa_psgc_ds_34_metadata',
    'psa_psgc_ds_34_psgc',
    'psa_psgc_ds_34_notes',
    'psa_psgc_ds_34_national_summary',
    'psa_psgc_ds_34_prov_sum'
  )
  AND target_table_name IS NOT NULL

In [ ]:
# DANGER ZONE: drops 8 tables from tuklas_dev.02_bronze.
# Runs only when DROP_CONFIRMED is set to True by hand.

DROP_CONFIRMED = False  # <-- set to True after checking the pre-flight and safety cells

CATALOG, SCHEMA = "tuklas_dev", "02_bronze"
STALE_TABLES = [
    "psa_psgc_report_text",
    "psa_psgc_source_files",
    "psa_psgc_ds_33_provincial_summary",
    "psa_psgc_ds_34_metadata",
    "psa_psgc_ds_34_psgc",
    "psa_psgc_ds_34_notes",
    "psa_psgc_ds_34_national_summary",
    "psa_psgc_ds_34_prov_sum",
]

if not DROP_CONFIRMED:
    print("Dry run only: nothing dropped. Set DROP_CONFIRMED = True to execute.")
    for t in STALE_TABLES:
        print(f"  would drop: {CATALOG}.{SCHEMA}.{t}")
else:
    failed = []
    for t in STALE_TABLES:
        full_name = f"{CATALOG}.`{SCHEMA}`.{t}"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_name}")
            print(f"Dropped: {full_name}")
        except Exception as e:
            failed.append(t)
            print(f"FAILED to drop {full_name}: {e}")
    if failed:
        raise RuntimeError(f"{len(failed)} table(s) not dropped: {failed}")
    print(f"Done. {len(STALE_TABLES)} tables dropped.")

In [ ]:
%sql
-- Post-check: 02_bronze should have 50 tables (58 before), and none of the stale names should remain.
SELECT
  (SELECT COUNT(*) FROM system.information_schema.tables
   WHERE table_catalog = 'tuklas_dev' AND table_schema = '02_bronze')                             AS bronze_table_count,
  (SELECT COUNT(*) FROM system.information_schema.tables
   WHERE table_catalog = 'tuklas_dev' AND table_schema = '02_bronze' AND table_name LIKE 'psa_psgc_%') AS stale_left,
  (SELECT COUNT(*) FROM system.information_schema.tables
   WHERE table_catalog = 'tuklas_dev' AND table_schema <> 'information_schema')                    AS tuklas_dev_table_count